# AI HW2: SimCLR Self-Supervised Learning on CIFAR-10

**Before running:** Go to `Runtime → Change runtime type → Hardware accelerator → T4 GPU`

---

### Notebook structure
| Section | What it does |
|---|---|
| 1. Setup | Check GPU, install/import libraries |
| 2. Config | All hyperparameters in one place |
| 3. Dataset | Data loading + SimCLR augmentations |
| 4. Model | Modified ResNet-18 + projector head |
| 5. Loss | NT-Xent contrastive loss |
| 6. Evaluate | kNN monitor + linear probing |
| 7. Trainer | Training loops |
| 8. Experiment 1 | SimCLR SSL training + linear probing |
| 9. Experiment 2 | Supervised learning baseline |

## 1. Setup

In [ ]:
# Verify GPU is available — must show a CUDA device, not CPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# (Optional) Mount Google Drive to save checkpoints and results permanently.
# If you skip this, files are saved locally in Colab and will be lost when the session ends.

USE_DRIVE = False   # Set to True to enable Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AI_HW2'
else:
    BASE_DIR = '/content/AI_HW2'

import os
os.makedirs(f'{BASE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{BASE_DIR}/results',     exist_ok=True)
print(f"Working directory: {BASE_DIR}")

In [ ]:
import sys
from datetime import datetime

class _Tee:
    """
    Mirrors all print() output to both the notebook and a persistent log file.
    Use as a context manager around any training or evaluation call:

        with _Tee(LOG_FILE):
            train_simclr(...)   # every print inside is saved automatically
    """
    def __init__(self, filepath):
        self._file   = open(filepath, 'a')
        self._stdout = sys.stdout
    def write(self, msg):
        self._stdout.write(msg)
        self._file.write(msg)
    def flush(self):
        self._stdout.flush()
        self._file.flush()
    def __enter__(self):
        sys.stdout = self
        return self
    def __exit__(self, *args):
        sys.stdout = self._stdout
        self._file.close()

LOG_FILE = f'{BASE_DIR}/results/training_log.txt'

# Create / reset the log file with a header
with open(LOG_FILE, 'w') as f:
    f.write(f"SimCLR Training Log\n")
    f.write(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n\n")

print(f"Log file ready: {LOG_FILE}")

In [ ]:
# Standard imports used throughout the notebook
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 2. Config — Hyperparameters

All settings are defined here. **Change values in this cell** to run different experiments (ablation studies) without touching any other code.

In [ ]:
# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
DATA_DIR    = '/content/data'
NUM_CLASSES = 10          # CIFAR-10 has 10 classes
NUM_WORKERS = 2           # Colab works best with 2 workers

# ---------------------------------------------------------------------------
# SSL Training (SimCLR)
# ---------------------------------------------------------------------------
SSL_EPOCHS      = 200
SSL_BATCH_SIZE  = 512     # T4 has 16 GB VRAM — 512 fits comfortably
SSL_LR          = 3e-4
SSL_WEIGHT_DECAY = 1e-6

# ---------------------------------------------------------------------------
# SimCLR-specific
# ---------------------------------------------------------------------------
# Temperature controls the sharpness of the similarity distribution.
# Try 0.1 (very sharp) or 5.0 (very flat) for ablation experiments.
TEMPERATURE          = 0.5
PROJECTOR_HIDDEN_DIM = 512
PROJECTOR_OUT_DIM    = 128

# ---------------------------------------------------------------------------
# kNN Monitor
# ---------------------------------------------------------------------------
KNN_K        = 20    # Number of nearest neighbors
KNN_INTERVAL = 5     # Run kNN every N epochs (increase to 10 to save time)
KNN_TEMP     = 0.1   # Temperature for kNN weighted voting

# ---------------------------------------------------------------------------
# Linear Probing
# ---------------------------------------------------------------------------
LP_EPOCHS       = 100
LP_LR           = 1e-3
LP_WEIGHT_DECAY = 1e-6

# ---------------------------------------------------------------------------
# Supervised Learning Baseline
# ---------------------------------------------------------------------------
SL_EPOCHS       = 200
SL_LR           = 3e-4
SL_WEIGHT_DECAY = 1e-6

print("Config loaded.")
print(f"  SSL: {SSL_EPOCHS} epochs, batch {SSL_BATCH_SIZE}, temp {TEMPERATURE}")
print(f"  kNN: k={KNN_K}, every {KNN_INTERVAL} epochs")
print(f"  Linear probe: {LP_EPOCHS} epochs")

## 3. Dataset & Augmentation

SimCLR requires **two different random augmentations** of the same image to form a positive pair. The `SimCLRTransform` class applies the same random pipeline twice, producing `(view1, view2)` from a single source image.

In [ ]:
# CIFAR-10 normalization constants (per-channel mean and std of the training set)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)


class SimCLRTransform:
    """
    Applies two independent random augmentations to the same image.
    Returns (view1, view2) — a positive pair for contrastive learning.
    """
    def __init__(self, image_size=32):
        self.augment = transforms.Compose([
            # Random crop forces the network to learn from partial views
            transforms.RandomResizedCrop(image_size, scale=(0.2, 1.0)),
            # Horizontal flip: learn orientation-invariant features
            transforms.RandomHorizontalFlip(p=0.5),
            # Color jitter: learn color-invariant features
            transforms.RandomApply([
                transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                       saturation=0.4, hue=0.1)
            ], p=0.8),
            # Random grayscale: prevent over-reliance on color
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
        ])

    def __call__(self, image):
        # Run the pipeline twice — randomness makes the two outputs look different
        return self.augment(image), self.augment(image)


# Evaluation transform: no augmentation, just convert and normalize
EVAL_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

# Supervised training transform: mild augmentation
SL_TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])


def get_ssl_loader(batch_size):
    dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True,
                                transform=SimCLRTransform())
    return DataLoader(dataset, batch_size=batch_size, shuffle=True,
                      num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)


def get_eval_loaders(batch_size):
    train_set = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=EVAL_TRANSFORM)
    test_set  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=EVAL_TRANSFORM)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, test_loader


def get_supervised_loaders(batch_size):
    train_set = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=SL_TRAIN_TRANSFORM)
    test_set  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=EVAL_TRANSFORM)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, test_loader


print("Dataset classes defined.")

## 4. Model Architecture

**Why modify ResNet-18?**  
The original ResNet-18 was designed for 224×224 ImageNet images. Its first layer downsamples aggressively (224→56). For CIFAR-10's 32×32 images, that same reduction would leave only 8×8 feature maps — too small. We replace the 7×7/stride-2 conv with a 3×3/stride-1 conv and remove the max-pool, keeping spatial size at 32×32 through the early layers.

In [ ]:
class ModifiedResNet18(nn.Module):
    """
    ResNet-18 adapted for 32x32 CIFAR-10 images.
    Changes: first conv 7x7/stride-2 → 3x3/stride-1; max-pool → Identity.
    Output: 512-dimensional feature vector per image.
    """
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=None)  # No pretrained weights

        # Replace first conv: keeps spatial size 32→32 instead of 32→16
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        # Remove max-pool (would further halve spatial size)
        resnet.maxpool = nn.Identity()
        # Remove the final classification layer — we want raw 512-dim features
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        self.output_dim = 512

    def forward(self, x):
        features = self.encoder(x)                      # [B, 512, 1, 1]
        return torch.flatten(features, start_dim=1)     # [B, 512]


class ProjectorHead(nn.Module):
    """
    Two-layer MLP: 512 → ReLU → 512 → 128.
    Used only during SSL training; discarded afterward.
    The loss is computed on this 128-dim output, not the backbone output.
    """
    def __init__(self, input_dim=512, hidden_dim=512, output_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.mlp(x)


class SimCLRModel(nn.Module):
    """Backbone + Projector Head. Use .encode() to get backbone features only."""
    def __init__(self):
        super().__init__()
        self.backbone  = ModifiedResNet18()
        self.projector = ProjectorHead(512, PROJECTOR_HIDDEN_DIM, PROJECTOR_OUT_DIM)

    def forward(self, x):
        h = self.backbone(x)    # 512-dim representation
        z = self.projector(h)   # 128-dim projection (for loss computation)
        return h, z

    def encode(self, x):
        return self.backbone(x)


class SupervisedModel(nn.Module):
    """Same backbone + a linear classification head. Trained with cross-entropy."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone   = ModifiedResNet18()
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def encode(self, x):
        return self.backbone(x)


# Quick sanity check: verify output shapes are correct
_dummy = torch.zeros(4, 3, 32, 32)
_m = SimCLRModel()
_h, _z = _m(_dummy)
print(f"SimCLRModel  — backbone output: {_h.shape}, projector output: {_z.shape}")
print(f"SupervisedModel — logits: {SupervisedModel()(_dummy).shape}")
del _dummy, _m, _h, _z

## 5. NT-Xent Loss

The contrastive loss that drives SimCLR. For each image, it asks: *"Given this augmented view, identify its twin among all 2N images in the batch."* This is cross-entropy where each image's "correct answer" is the index of its positive pair.

In [ ]:
class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy Loss (SimCLR paper, Chen et al. 2020).

    Given N positive pairs (z_i[k], z_j[k]):
      1. Normalize all 2N vectors to unit length
      2. Compute 2N x 2N cosine similarity matrix, scaled by 1/temperature
      3. Mask out self-similarity (diagonal)
      4. For each of the 2N vectors, cross-entropy loss treats its paired
         twin as the "correct class" and all others as "wrong classes"
      5. Return the mean loss over all 2N vectors
    """
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i, z_j):
        N = z_i.shape[0]

        # L2-normalize so dot product = cosine similarity
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)

        # Stack into [2N, D] — first N rows are z_i, next N rows are z_j
        z = torch.cat([z_i, z_j], dim=0)

        # Full pairwise similarity matrix [2N, 2N], scaled by temperature
        sim = torch.mm(z, z.T) / self.temperature

        # Positive pair labels:
        #   row i (from z_i) → positive at row i+N (in z_j)
        #   row i+N (from z_j) → positive at row i (in z_i)
        labels = torch.arange(N, device=z_i.device)
        labels = torch.cat([labels + N, labels])

        # Mask self-similarity: set diagonal to -inf so softmax ignores it
        mask = torch.eye(2 * N, dtype=torch.bool, device=z_i.device)
        sim  = sim.masked_fill(mask, float('-inf'))

        return F.cross_entropy(sim, labels)


print("NTXentLoss defined.")

## 6. Evaluation Utilities

**kNN Monitor** — runs during SSL training to track whether the backbone is learning useful features, without needing labels.

**Linear Probing** — run once after SSL training is complete. Freeze the backbone, train a single linear layer on labeled data.

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    """Run all images through model.encode() and return (features, labels)."""
    model.eval()
    feats, labs = [], []
    for images, labels in loader:
        feats.append(model.encode(images.to(DEVICE)).cpu())
        labs.append(labels)
    return torch.cat(feats), torch.cat(labs)


@torch.no_grad()
def knn_monitor(model, train_loader, test_loader, k=20, temperature=0.1, num_classes=10):
    """
    kNN classification accuracy — measures representation quality without any training.
    For each test image, find its k most similar training images and predict by weighted vote.
    """
    train_feats, train_labels = extract_features(model, train_loader)
    test_feats,  test_labels  = extract_features(model, test_loader)

    # Normalize for cosine similarity
    train_feats = F.normalize(train_feats, dim=1).to(DEVICE)
    test_feats  = F.normalize(test_feats,  dim=1).to(DEVICE)
    train_labels = train_labels.to(DEVICE)

    correct = total = 0
    for start in range(0, len(test_feats), 512):
        end         = min(start + 512, len(test_feats))
        batch_feats = test_feats[start:end]
        batch_labs  = test_labels[start:end].to(DEVICE)

        # Cosine similarity between this test batch and all training images
        sim = torch.mm(batch_feats, train_feats.T) / temperature  # [chunk, N_train]

        # Top-k neighbors
        topk_vals, topk_idx = sim.topk(k, dim=1)
        weights      = F.softmax(topk_vals, dim=1)          # [chunk, k]
        neighbor_labs = train_labels[topk_idx]              # [chunk, k]

        # Weighted vote per class
        votes = torch.zeros(end - start, num_classes, device=DEVICE)
        votes.scatter_add_(1, neighbor_labs, weights)

        correct += (votes.argmax(dim=1) == batch_labs).sum().item()
        total   += len(batch_labs)

    return correct / total


def linear_probing(model, train_loader, test_loader,
                   num_classes=10, epochs=100, lr=1e-3, weight_decay=1e-6):
    """
    Train a single linear layer on top of frozen backbone features.
    Returns final test accuracy.
    """
    # Freeze backbone — no gradients will flow through it
    for p in model.backbone.parameters():
        p.requires_grad = False
    model.eval()

    linear    = nn.Linear(model.backbone.output_dim, num_classes).to(DEVICE)
    optimizer = Adam(linear.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn   = nn.CrossEntropyLoss()

    print(f"  [Linear Probing] {epochs} epochs (backbone frozen)")
    for epoch in range(1, epochs + 1):
        linear.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.no_grad():
                features = model.encode(images)
            loss = loss_fn(linear(features), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 20 == 0:
            acc = _eval_linear(linear, model, test_loader)
            print(f"    Epoch {epoch:3d}/{epochs}  loss={total_loss/len(train_loader):.4f}  "
                  f"test_acc={acc*100:.2f}%")

    final_acc = _eval_linear(linear, model, test_loader)
    print(f"  Final Linear Probing Accuracy: {final_acc*100:.2f}%")

    # Unfreeze backbone for future use
    for p in model.backbone.parameters():
        p.requires_grad = True

    return final_acc


@torch.no_grad()
def _eval_linear(linear, model, loader):
    linear.eval(); model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (linear(model.encode(images)).argmax(1) == labels).sum().item()
        total   += len(labels)
    return correct / total


@torch.no_grad()
def evaluate_accuracy(model, loader):
    """Test accuracy of a model whose forward() returns class logits."""
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(1) == labels).sum().item()
        total   += len(labels)
    return correct / total


print("Evaluation utilities defined.")

## 7. Training Functions

In [ ]:
def train_simclr(model, ssl_loader, eval_train_loader, eval_test_loader):
    """
    SimCLR SSL training loop.
    Labels are never used — the only training signal is the contrastive loss.
    Returns history dict with 'loss' and 'knn_acc' lists.
    """
    optimizer = Adam(model.parameters(), lr=SSL_LR, weight_decay=SSL_WEIGHT_DECAY)
    criterion = NTXentLoss(temperature=TEMPERATURE)
    history   = {'loss': [], 'knn_acc': []}

    print(f"\n{'='*60}")
    print(f"SimCLR SSL Training: {SSL_EPOCHS} epochs, "
          f"batch {SSL_BATCH_SIZE}, temp {TEMPERATURE}")
    print(f"{'='*60}")

    for epoch in range(1, SSL_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for (view1, view2), _ in ssl_loader:
            # Two augmented views of the same images; labels (_) are ignored
            view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
            _, z_i = model(view1)   # projector outputs, 128-dim
            _, z_j = model(view2)
            loss = criterion(z_i, z_j)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(ssl_loader)
        history['loss'].append(avg_loss)

        # kNN monitor every KNN_INTERVAL epochs
        run_knn = (epoch % KNN_INTERVAL == 0)
        if run_knn:
            acc = knn_monitor(model, eval_train_loader, eval_test_loader,
                              k=KNN_K, temperature=KNN_TEMP, num_classes=NUM_CLASSES)
            history['knn_acc'].append((epoch, acc))

        if epoch % 10 == 0 or epoch == 1:
            knn_str = f"  kNN: {acc*100:.2f}%" if run_knn and epoch % 10 == 0 else ""
            print(f"  Epoch {epoch:3d}/{SSL_EPOCHS}  loss={avg_loss:.4f}{knn_str}")

    print("SSL Training complete.")
    return history


def train_supervised(model, train_loader, test_loader):
    """
    Standard supervised classification training loop.
    Uses cross-entropy loss with ground-truth labels.
    Returns history dict with 'loss' and 'test_acc' lists.
    """
    optimizer = Adam(model.parameters(), lr=SL_LR, weight_decay=SL_WEIGHT_DECAY)
    loss_fn   = nn.CrossEntropyLoss()
    history   = {'loss': [], 'test_acc': []}

    print(f"\n{'='*60}")
    print(f"Supervised Learning: {SL_EPOCHS} epochs, batch {SSL_BATCH_SIZE}")
    print(f"{'='*60}")

    for epoch in range(1, SL_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            loss = loss_fn(model(images), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        history['loss'].append(avg_loss)

        if epoch % 10 == 0 or epoch == 1:
            acc = evaluate_accuracy(model, test_loader)
            history['test_acc'].append((epoch, acc))
            print(f"  Epoch {epoch:3d}/{SL_EPOCHS}  "
                  f"loss={avg_loss:.4f}  test_acc={acc*100:.2f}%")

    print("Supervised Training complete.")
    return history


print("Training functions defined.")

## 8. Experiment 1: SimCLR SSL Training + Linear Probing

This is the main experiment. The backbone learns representations **without labels** via contrastive loss. Afterward, we evaluate how good those representations are using linear probing.

In [ ]:
# Prepare data loaders
ssl_loader             = get_ssl_loader(SSL_BATCH_SIZE)
eval_train_loader, \
eval_test_loader       = get_eval_loaders(SSL_BATCH_SIZE)

print("Data loaders ready.")

In [ ]:
ssl_model = SimCLRModel().to(DEVICE)

with _Tee(LOG_FILE):
    print("=== Experiment 1: SimCLR SSL Training ===\n")
    ssl_history = train_simclr(ssl_model, ssl_loader, eval_train_loader, eval_test_loader)
    torch.save(ssl_model.state_dict(), f'{BASE_DIR}/checkpoints/simclr.pt')
    print("Checkpoint saved.")

In [ ]:
# Plot SSL training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"SimCLR SSL Training (temp={TEMPERATURE}, batch={SSL_BATCH_SIZE})", fontsize=13)

axes[0].plot(ssl_history['loss'], color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('NT-Xent Loss')
axes[0].set_title('Training Loss'); axes[0].grid(alpha=0.3)

if ssl_history['knn_acc']:
    epochs, accs = zip(*ssl_history['knn_acc'])
    axes[1].plot(epochs, [a*100 for a in accs], color='darkorange', marker='o', markersize=3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('kNN Accuracy (%)')
    axes[1].set_title(f'kNN Monitor (k={KNN_K})'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/results/ssl_curves.png', dpi=150)
plt.show()
print("Plot saved.")

In [ ]:
with _Tee(LOG_FILE):
    print("\n=== Experiment 1: Linear Probing ===\n")
    ssl_lp_acc = linear_probing(
        ssl_model, eval_train_loader, eval_test_loader,
        num_classes=NUM_CLASSES, epochs=LP_EPOCHS, lr=LP_LR, weight_decay=LP_WEIGHT_DECAY
    )

## 9. Experiment 2: Supervised Learning Baseline

Same backbone architecture, but trained end-to-end with labeled data and cross-entropy loss. Compare its test accuracy against SSL + linear probing from Experiment 1.

In [ ]:
sl_train_loader, sl_test_loader = get_supervised_loaders(SSL_BATCH_SIZE)
sl_model = SupervisedModel(num_classes=NUM_CLASSES).to(DEVICE)

with _Tee(LOG_FILE):
    print("\n=== Experiment 2: Supervised Learning ===\n")
    sl_history = train_supervised(sl_model, sl_train_loader, sl_test_loader)
    torch.save(sl_model.state_dict(), f'{BASE_DIR}/checkpoints/supervised.pt')
    print("Checkpoint saved.")

In [ ]:
# Plot supervised training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Supervised Learning Training', fontsize=13)

axes[0].plot(sl_history['loss'], color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss'); axes[0].grid(alpha=0.3)

if sl_history['test_acc']:
    epochs, accs = zip(*sl_history['test_acc'])
    axes[1].plot(epochs, [a*100 for a in accs], color='darkorange', marker='o', markersize=3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy (%)')
    axes[1].set_title('Test Accuracy'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/results/sl_curves.png', dpi=150)
plt.show()

In [ ]:
sl_final_acc = evaluate_accuracy(sl_model, sl_test_loader)

with _Tee(LOG_FILE):
    print("\n" + "="*50)
    print("  RESULTS SUMMARY")
    print("="*50)
    try:
        print(f"  SSL + Linear Probing : {ssl_lp_acc*100:.2f}%")
    except NameError:
        print("  SSL + Linear Probing : (not run)")
    try:
        print(f"  Supervised Learning  : {sl_final_acc*100:.2f}%")
    except NameError:
        print("  Supervised Learning  : (not run)")
    print("="*50)

# Save numeric summary to a separate clean file for easy copy-paste into report
lines = [f"Temperature         : {TEMPERATURE}\n", f"Batch size          : {SSL_BATCH_SIZE}\n"]
try:
    lines.append(f"SSL+LinearProbe acc : {ssl_lp_acc*100:.2f}%\n")
except NameError:
    pass
try:
    lines.append(f"Supervised acc      : {sl_final_acc*100:.2f}%\n")
except NameError:
    pass

with open(f'{BASE_DIR}/results/summary.txt', 'w') as f:
    f.writelines(lines)
print(f"Summary saved to {BASE_DIR}/results/summary.txt")

## 10. Experiment 3: Random Frozen Backbone (lower bound) — Optional

This baseline shows what accuracy you get when the backbone is **never trained at all** — its weights stay random forever, and only the linear probing layer is trained.

It answers: *"How much does SSL training actually help compared to doing nothing?"*

```
Random Frozen  <  SSL + Linear Probing  <  Supervised Learning
  (lower bound)                              (upper bound)
```

Run this cell independently — no need to re-run Experiments 1 or 2.

In [ ]:
# Create eval loaders if Experiment 1 hasn't been run yet
try:
    eval_train_loader, eval_test_loader
except NameError:
    eval_train_loader, eval_test_loader = get_eval_loaders(SSL_BATCH_SIZE)
    print("Eval loaders created.")

random_model = SimCLRModel().to(DEVICE)

with _Tee(LOG_FILE):
    print("\n=== Experiment 3: Random Frozen Backbone (lower bound) ===\n")
    random_lp_acc = linear_probing(
        random_model, eval_train_loader, eval_test_loader,
        num_classes=NUM_CLASSES, epochs=LP_EPOCHS, lr=LP_LR, weight_decay=LP_WEIGHT_DECAY
    )
    print("\n" + "="*50)
    print("  FULL COMPARISON")
    print("="*50)
    try:
        print(f"  Random Frozen Backbone : {random_lp_acc*100:.2f}%  ← lower bound")
        print(f"  SSL + Linear Probing   : {ssl_lp_acc*100:.2f}%")
        print(f"  Supervised Learning    : {sl_final_acc*100:.2f}%  ← upper bound")
    except NameError:
        print(f"  Random Frozen Backbone : {random_lp_acc*100:.2f}%")
        print("  (Run Experiments 1 & 2 to see the full comparison)")
    print("="*50)